# 4단계: Q1 넷째 데이터셋 — MVTec AD 2

Q1 과 같은 설정 10개 x 시드 3개를 MVTec AD 2 의 8부류에 돌려, 세 데이터셋에서 낸 순위와 견준다.

| 갈래 | 장수 | 쓰나 |
|---|---|---|
| train/good | 2,528 (부류마다 137~432), 모두 regular | 학습 |
| test_public | 1,084 (정상 379, 결함 705, 마스크 705) | 채점 |
| validation/good | 302 | 안 씀(두 방법 모두 쓰지 않음) |
| test_private, test_private_mixed | 부류마다 142~321 | 안 씀(정답 비공개) |

**정한 것(2026-11-13).**
- 전처리는 Q1 과 같은 `Resize((256, 256))`, 자르지 않음. 원본이 1400~4224 픽셀이라 256 에서 결함 덩어리 2,058개 중 483개가 사라지고, 마스크 12장(can 6, fabric 6)은 비게 된다(mvtec2_check2 노트북). 이미지 라벨은 남으므로 이미지 AUROC 를 주 지표로 본다.
- test_public 의 조명 조건(regular, overexposed, underexposed, shift_1~3, vial 만 shift_4)을 모두 함께 채점한다. 조건별 이미지 AUROC 는 5번 둘째 셀에서 점수 파일로 따로 낸다. 학습은 regular 만 있어서, 순위가 바뀌면 그 안에 조명 변화에 대한 강건성이 섞인다.

**시간(어림, 잰 값 아님).**
- 2번 캐시: 원본을 256 으로 한 번 줄여 Drive `mvtec2_256/` 에 둔다. 부류마다 10장으로 잰 첫 읽기 속도(평균 4.4초, fabric 19.8초)로 4,317장이면 약 6시간이고, 여기서는 8장씩 동시에 읽으니 그보다 짧을 수 있다. 한 번 만들면 다시 만들지 않는다.
- 4번 실행: 240번, Q1 의 MVTec AD 몫(450번, 5.9시간)을 장수 비로 줄여 약 5.1시간.

**끊겨도 된다.**
- 2번 캐시는 한 장마다 Drive 에 저장한다. 다시 돌리면 있는 파일은 건너뛰고, 다 끝난 부류는 통째로 건너뛴다.
- 4번 실행은 부류 하나가 끝날 때마다 Drive 의 `q1_mvtec_ad_2.csv` 에 한 줄을 적는다. 다시 돌리면 적힌 줄은 건너뛴다.
- 이어 돌리는 순서: 런타임 > 세션 다시 시작 -> 1번 셀 -> 2번 두 셀(캐시가 끝났으면 바로 지나간다) -> 3번 첫 셀 -> 4번 셀.

**실행 전 준비.** 런타임 GPU(L4), 보안 비밀 `GITHUB_TOKEN` 접근 허용. 02 노트북과 같다.


## 1. 코드와 패키지

In [1]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
REPO_URL = "https://github.com/chlee1027/beyond-mvtec.git"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN").strip()
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", auth_url, REPO_DIR], capture_output=True, text=True)
else:
    # 이미 받아 둔 레포는 GitHub 의 최신 main 으로 맞춘다(토큰 든 주소는 저장하지 않는다)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, "main"],
                       capture_output=True, text=True)
# 실패해도 오류 메시지에 토큰이 찍히지 않게 가린다
assert r.returncode == 0, "레포 받기 실패: " + r.stderr.replace(token, "***")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

from beyond_mvtec import data, experiment as ex, patchcore_adapter
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
SETTINGS = ex.comparison_settings(config)
# 레포가 MVTec AD 2 코드보다 옛 판이면 여기서 멈춘다
assert len(SETTINGS) == 10 and hasattr(data, "list_mvtec2") and hasattr(data, "cache_mvtec2") \
    and "mvtec_ad_2" in config["datasets"] and hasattr(patchcore_adapter, "_releasing_sampler"), \
    "옛 코드가 읽혔다. 런타임 > 세션 다시 시작 후 1번 셀부터 다시 돌린다"
print("설정", len(SETTINGS), "개:", [s["setting"] for s in SETTINGS])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
레포 b565e86 | Python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | faiss 1.14.1
GPU NVIDIA L4 | VRAM 23.7 GB
설정 10 개: ['patchcore_r18_L23_c1', 'patchcore_r18_L23_c10', 'patchcore_r18_L123_c1', 'patchcore_wrn50_L23_c1', 'patchcore_wrn50_L23_c10', 'patchcore_wrn50_L123_c1', 'padim_r18_L123_d100', 'padim_r18_L23_d100', 'padim_wrn50_L123_d550', 'padim_wrn50_L23_d550']


## 2. 256 캐시와 데이터

첫 셀이 Drive `mvtec2/` 의 원본을 `mvtec2_256/` 으로 줄여 저장하고(부류마다 진행 표시), 다 된 캐시를 런타임 디스크로 옮긴다. 둘째 셀은 장수와 조건별 장수를 점검 노트북에서 센 값과 맞춰 보고, 부류마다 원본 세 장을 직접 읽은 텐서와 캐시를 읽은 텐서가 같은지 본다.

In [2]:
import shutil, time
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

PREPROCESS = config["preprocess"]["compare"]
SIZE = config["datasets"]["mvtec_ad_2"]["cache"]
# 캐시 크기와 실행 전처리가 같아야 원본을 읽은 것과 값이 같다
assert list(PREPROCESS["resize"]) == list(SIZE) and not PREPROCESS.get("center_crop"), (PREPROCESS, SIZE)

SRC_DRIVE = f"{MY}/mvtec2"
CACHE_DRIVE = f"{MY}/mvtec2_256"
LOCAL = "/content/data/mvtec_ad_2"
CATEGORIES = data.MVTEC2_CATEGORIES
assert all(os.path.isdir(f"{SRC_DRIVE}/{c}") for c in CATEGORIES), os.listdir(SRC_DRIVE)

for category in CATEGORIES:
    start = time.time()
    n = data.cache_mvtec2(SRC_DRIVE, CACHE_DRIVE, category, SIZE, workers=8)
    print(f"{category}: 새로 쓴 장수 {n}, {time.time() - start:.0f}초")

if not os.path.isfile(f"{LOCAL}/.copied"):
    shutil.copytree(CACHE_DRIVE, LOCAL, dirs_exist_ok=True)
    open(f"{LOCAL}/.copied", "w").close()
print("Drive:", CACHE_DRIVE, "-> 로컬:", LOCAL)


Mounted at /content/drive
can: 200/664
can: 400/664
can: 600/664
can: 664/664
can: 새로 쓴 장수 664, 62초
fabric: 200/633
fabric: 400/633
fabric: 600/633
fabric: 633/633
fabric: 새로 쓴 장수 633, 89초
fruit_jelly: 200/403
fruit_jelly: 400/403
fruit_jelly: 403/403
fruit_jelly: 새로 쓴 장수 403, 40초
rice: 200/535
rice: 400/535
rice: 535/535
rice: 새로 쓴 장수 535, 71초
sheet_metal: 200/341
sheet_metal: 341/341
sheet_metal: 새로 쓴 장수 341, 48초
vial: 200/536
vial: 400/536
vial: 536/536
vial: 새로 쓴 장수 536, 47초
wallplugs: 200/533
wallplugs: 400/533
wallplugs: 533/533
wallplugs: 새로 쓴 장수 533, 51초
walnuts: 200/672
walnuts: 400/672
walnuts: 600/672
walnuts: 672/672
walnuts: 새로 쓴 장수 672, 74초
Drive: /content/drive/MyDrive/mvtec2_256 -> 로컬: /content/data/mvtec_ad_2


In [3]:
import numpy as np
import pandas as pd
import torch

rows, conds = [], []
for category in CATEGORIES:
    split = data.list_mvtec2(LOCAL, category)
    rows.append({"category": category, "train": len(split.train), "test": len(split.test),
                 "test_anomaly": sum(s.label for s in split.test)})
    conds += [{"category": category, "label": s.label, "condition": s.anomaly} for s in split.test]
counts = pd.DataFrame(rows).set_index("category")
display(counts)

# mvtec2_check · mvtec2_check2 노트북에서 센 값
expected = {"can": (412, 162, 90), "fabric": (387, 156, 90), "fruit_jelly": (263, 80, 60),
            "rice": (313, 132, 90), "sheet_metal": (137, 114, 90), "vial": (291, 140, 105),
            "wallplugs": (293, 150, 90), "walnuts": (432, 150, 90)}
got = {c: tuple(int(v) for v in r) for c, r in counts[["train", "test", "test_anomaly"]].iterrows()}
assert got == expected, got
display(pd.DataFrame(conds).pivot_table(index="category", columns="condition", values="label",
                                        aggfunc="size", fill_value=0))

# 캐시를 읽은 텐서가 원본을 읽은 텐서와 같은지, 부류마다 학습 한 장 · 결함 한 장 · 그 마스크로 본다
image_tf, mask_tf, output_size = data.build_transforms(PREPROCESS)
for category in CATEGORIES:
    orig, cached = data.list_mvtec2(SRC_DRIVE, category), data.list_mvtec2(LOCAL, category)
    a = data.TrainDataset(orig.train[:1], image_tf)[0]["image"]
    b = data.TrainDataset(cached.train[:1], image_tf)[0]["image"]
    i = next(k for k, s in enumerate(orig.test) if s.label == 1)
    ta = data.TestDataset(orig.test[i:i + 1], image_tf, mask_tf, output_size)[0]
    tb = data.TestDataset(cached.test[i:i + 1], image_tf, mask_tf, output_size)[0]
    same = torch.equal(a, b) and torch.equal(ta["image"], tb["image"]) and torch.equal(ta["mask"], tb["mask"])
    print(category, "같다" if same else "다르다")
    assert same, category


,train,test,test_anomaly
category,,,
can,412,162,90
fabric,387,156,90
fruit_jelly,263,80,60
rice,313,132,90
sheet_metal,137,114,90
vial,291,140,105
wallplugs,293,150,90
walnuts,432,150,90


condition,overexposed,regular,shift_1,shift_2,shift_3,shift_4,underexposed
category,,,,,,,
can,27,27,27,27,27,0,27
fabric,26,26,26,26,26,0,26
fruit_jelly,20,20,20,0,0,0,20
rice,22,22,22,22,22,0,22
sheet_metal,19,19,19,19,19,0,19
vial,20,20,20,20,20,20,20
wallplugs,25,25,25,25,25,0,25
walnuts,25,25,25,25,25,0,25


can 같다
fabric 같다
fruit_jelly 같다
rice 같다
sheet_metal 같다
vial 같다
wallplugs 같다
walnuts 같다


## 3. 짧은 실행: 가장 무거운 칸 먼저

학습이 가장 많은 `walnuts`(432장), 시드 0에서 두 설정을 먼저 돈다. 결과는 4번과 같은 CSV에 적히므로 4번에서 다시 돌지 않는다.

- `patchcore_wrn50_L123_c1`: Q1 에서 MVTec AD `hazelnut`(391장)의 `fit_sec` 가 204.6~208.3초였다. 코어셋 시간은 장수의 제곱에 비례하므로 432장이면 약 252초로 어림한다.
- `padim_wrn50_L123_d550`: 같은 256 해상도라 위치 수가 같다. Q1 MVTec AD 최대 GPU 메모리는 17.83 GB였다.

**넘어가는 기준.** 둘 다 오류 없이 끝나고, PatchCore의 `fit_sec`가 200~460초 안이면(어림의 0.8~1.8배, 02 노트북과 같은 폭) 4번으로 바로 간다. 메모리 오류가 나거나 `fit_sec`가 이 범위를 벗어나면 4번을 돌리지 말고 이 셀 출력을 보낸다.

In [4]:
import resource

OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT,
               save_dir=f"{RESULT_DIR}/q1_scores")
CSV = f"{RESULT_DIR}/q1_mvtec_ad_2.csv"


In [5]:
heavy = [s for s in SETTINGS if s["setting"] in ("patchcore_wrn50_L123_c1", "padim_wrn50_L123_d550")]
ex.run_grid(heavy, "mvtec_ad_2", LOCAL, ["walnuts"], [0], "compare", PREPROCESS, CSV, **OPTIONS)

done = pd.read_csv(CSV)
display(done[(done.category == "walnuts") & (done.seed == 0) & done.setting.isin([s["setting"] for s in heavy])]
        [["setting", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]])
print(f"호스트 메모리 최대 {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 17694/17694 [02:35<00:00, 113.66it/s]


[1/2] patchcore_wrn50_L123_c1 walnuts s0 I=0.8589 P=0.9582 PRO=0.6067 fit=222.95s pred=25.16s gpu=8.5GB
[2/2] padim_wrn50_L123_d550 walnuts s0 I=0.8181 P=0.9188 PRO=0.5498 fit=26.71s pred=2.01s gpu=17.83GB


,setting,image_auroc,pixel_auroc,pro,bank_size,fit_sec,predict_sec,metric_sec,peak_gpu_gb
0,patchcore_wrn50_L123_c1,0.858889,0.958224,0.606697,17694.0,222.95,25.16,5.57,8.50
1,padim_wrn50_L123_d550,0.818148,0.918761,0.549774,NaN,26.71,2.01,5.49,17.83


호스트 메모리 최대 24.1 GB


## 4. 실행

시드 0 -> 1 -> 2 순서로, 시드마다 설정 10개 x 8부류를 돈다. 점수와 라벨은 `q1_scores/`에 남기고(5번 조건별 표에 쓴다) 이상 지도는 남기지 않는다.

In [6]:
for seed in config["seeds"]:
    print(f"===== 시드 {seed} | mvtec_ad_2 =====")
    ex.run_grid(SETTINGS, "mvtec_ad_2", LOCAL, CATEGORIES, [seed], "compare", PREPROCESS, CSV, **OPTIONS)


===== 시드 0 | mvtec_ad_2 =====


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 4218/4218 [00:09<00:00, 462.04it/s]


[1/80] patchcore_r18_L23_c1 can s0 I=0.5198 P=0.6053 PRO=0.2185 fit=28.68s pred=6.14s gpu=2.0GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 3962/3962 [00:08<00:00, 491.77it/s]


[2/80] patchcore_r18_L23_c1 fabric s0 I=0.5268 P=0.6788 PRO=0.2650 fit=26.41s pred=6.02s gpu=1.88GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 2693/2693 [00:03<00:00, 710.75it/s]


... (277 outputs left out)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 12001/12001 [01:13<00:00, 162.23it/s]


[47/80] patchcore_wrn50_L123_c1 wallplugs s2 I=0.4859 P=0.8353 PRO=0.4205 fit=130.21s pred=22.11s gpu=5.85GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 17694/17694 [02:40<00:00, 110.55it/s]


[48/80] patchcore_wrn50_L123_c1 walnuts s2 I=0.8728 P=0.9604 PRO=0.6094 fit=242.88s pred=23.0s gpu=8.5GB
[49/80] padim_r18_L123_d100 can s2 I=0.4901 P=0.8328 PRO=0.3918 fit=1.51s pred=1.42s gpu=0.65GB
[50/80] padim_r18_L123_d100 fabric s2 I=0.6315 P=0.7156 PRO=0.2173 fit=1.42s pred=1.37s gpu=0.65GB
[51/80] padim_r18_L123_d100 fruit_jelly s2 I=0.8908 P=0.9453 PRO=0.6612 fit=1.24s pred=1.05s gpu=0.66GB
[52/80] padim_r18_L123_d100 rice s2 I=0.5437 P=0.7726 PRO=0.4855 fit=1.36s pred=1.31s gpu=0.64GB
[53/80] padim_r18_L123_d100 sheet_metal s2 I=0.6333 P=0.7847 PRO=0.1447 fit=1.03s pred=1.16s gpu=0.64GB
[54/80] padim_r18_L123_d100 vial s2 I=0.7320 P=0.9037 PRO=0.8398 fit=1.26s pred=1.3s gpu=0.65GB
[55/80] padim_r18_L123_d100 wallplugs s2 I=0.3535 P=0.7782 PRO=0.2787 fit=1.31s pred=1.39s gpu=0.66GB
[56/80] padim_r18_L123_d100 walnuts s2 I=0.5981 P=0.9147 PRO=0.5225 fit=1.51s pred=1.34s gpu=0.67GB
[57/80] padim_r18_L23_d100 can s2 I=0.4870 P=0.7775 PRO=0.3116 fit=1.18s pred=1.36s gpu=0.27GB
[5

## 5. 순위와 순위 상관

레포 `results/` 의 Q1 세 CSV 와 이번 CSV 를 함께 읽어, 데이터셋 넷의 순위와 여섯 쌍의 Kendall tau-b 를 낸다. 시드 0만 끝난 상태에서도 돌릴 수 있다.

In [7]:
paths = [f"{REPO_DIR}/results/q1_{d}.csv" for d in ["mvtec_ad", "real_iad", "visa"]] + [CSV]
summary = ex.summarize(paths)
new = summary[summary.dataset == "mvtec_ad_2"]
display(new[["setting", "image_auroc_mean", "image_auroc_std", "pixel_auroc_mean", "pro_mean",
             "n_seeds", "n_categories"]].round(2))
print("칸이 다 찼나:", ((new.n_seeds == 3) & (new.n_categories == len(CATEGORIES))).sum() == len(SETTINGS))

values, ranks = ex.rank_by_dataset(paths)
print("이미지 AUROC (%, 시드 평균)")
display(values.round(2))
print("순위 (1 = 가장 높음)")
display(ranks)
print("Kendall tau-b, 시드 평균 값으로")
display(ex.kendall_pairs(values).round(3))
print("Kendall tau-b, 시드 하나씩")
display(ex.kendall_per_seed(paths).round(3))

df = pd.read_csv(CSV)
display(df.groupby("setting")[["fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))
print("더한 시간(시간):", round((df.fit_sec + df.predict_sec + df.metric_sec).sum() / 3600, 1))


,setting,image_auroc_mean,image_auroc_std,pixel_auroc_mean,pro_mean,n_seeds,n_categories
10,padim_r18_L123_d100,64.50,3.85,82.20,43.19,3,8
11,padim_r18_L23_d100,62.66,3.53,80.48,42.91,3,8
12,padim_wrn50_L123_d550,63.82,2.18,84.66,46.35,3,8
13,padim_wrn50_L23_d550,65.28,1.29,84.08,44.88,3,8
14,patchcore_r18_L123_c1,66.83,0.80,78.69,47.63,3,8
15,patchcore_r18_L23_c1,66.36,0.68,78.46,42.36,3,8
16,patchcore_r18_L23_c10,63.66,1.38,81.82,46.42,3,8
17,patchcore_wrn50_L123_c1,68.27,1.98,83.47,50.27,3,8
18,patchcore_wrn50_L23_c1,69.71,1.70,83.60,46.85,3,8
19,patchcore_wrn50_L23_c10,71.08,0.79,85.82,47.76,3,8


칸이 다 찼나: True
이미지 AUROC (%, 시드 평균)


dataset,mvtec_ad,mvtec_ad_2,real_iad,visa
setting,,,,
padim_r18_L123_d100,90.96,64.50,81.62,83.60
padim_r18_L23_d100,91.56,62.66,83.41,84.46
padim_wrn50_L123_d550,95.15,63.82,88.03,87.96
padim_wrn50_L23_d550,95.47,65.28,88.67,87.77
patchcore_r18_L123_c1,95.01,66.83,92.11,95.85
patchcore_r18_L23_c1,97.01,66.36,89.03,93.23
patchcore_r18_L23_c10,96.87,63.66,90.39,93.10
patchcore_wrn50_L123_c1,98.20,68.27,93.55,96.60
patchcore_wrn50_L23_c1,98.37,69.71,91.83,93.64


순위 (1 = 가장 높음)


dataset,mvtec_ad,mvtec_ad_2,real_iad,visa
setting,,,,
padim_r18_L123_d100,10.0,7.0,10.0,10.0
padim_r18_L23_d100,9.0,10.0,9.0,9.0
padim_wrn50_L123_d550,7.0,8.0,8.0,7.0
padim_wrn50_L23_d550,6.0,6.0,7.0,8.0
patchcore_r18_L123_c1,8.0,4.0,3.0,2.0
patchcore_r18_L23_c1,4.0,5.0,6.0,4.0
patchcore_r18_L23_c10,5.0,9.0,5.0,6.0
patchcore_wrn50_L123_c1,3.0,3.0,1.0,1.0
patchcore_wrn50_L23_c1,2.0,2.0,4.0,3.0


Kendall tau-b, 시드 평균 값으로


,pair,n_settings,n_pairs,concordant,discordant,tied,tau_b,p_value
0,mvtec_ad - mvtec_ad_2,10,45,36,9,0,0.600,0.017
1,mvtec_ad - real_iad,10,45,37,8,0,0.644,0.009
2,mvtec_ad - visa,10,45,34,11,0,0.511,0.047
3,mvtec_ad_2 - real_iad,10,45,36,9,0,0.600,0.017
4,mvtec_ad_2 - visa,10,45,33,12,0,0.467,0.073
5,real_iad - visa,10,45,40,5,0,0.778,0.001


Kendall tau-b, 시드 하나씩


,pair,n_settings,n_pairs,concordant,discordant,tied,tau_b,p_value,seed
0,mvtec_ad - mvtec_ad_2,10,45,40,5,0,0.778,0.001,0
1,mvtec_ad - real_iad,10,45,38,7,0,0.689,0.005,0
2,mvtec_ad - visa,10,45,36,9,0,0.600,0.017,0
3,mvtec_ad_2 - real_iad,10,45,35,10,0,0.556,0.029,0
4,mvtec_ad_2 - visa,10,45,35,10,0,0.556,0.029,0
5,real_iad - visa,10,45,41,4,0,0.822,0.000,0
6,mvtec_ad - mvtec_ad_2,10,45,31,14,0,0.378,0.156,1
7,mvtec_ad - real_iad,10,45,36,9,0,0.600,0.017,1
8,mvtec_ad - visa,10,45,34,11,0,0.511,0.047,1
9,mvtec_ad_2 - real_iad,10,45,30,15,0,0.333,0.216,1


,fit_sec,predict_sec,metric_sec,peak_gpu_gb
setting,,,,
padim_r18_L123_d100,1.3,1.3,5.1,0.7
padim_r18_L23_d100,1.0,1.2,5.1,0.3
padim_wrn50_L123_d550,18.1,1.9,5.0,17.8
padim_wrn50_L23_d550,5.2,1.5,5.0,6.5
patchcore_r18_L123_c1,149.4,18.3,5.0,6.0
patchcore_r18_L23_c1,21.1,5.2,5.0,1.5
patchcore_r18_L23_c10,74.8,6.1,5.0,1.5
patchcore_wrn50_L123_c1,151.6,19.9,5.0,6.5
patchcore_wrn50_L23_c1,22.7,5.9,5.0,1.8


더한 시간(시간): 4.3


In [8]:
# 조건별 이미지 AUROC: 그 조건의 정상과 결함만으로 낸다. 부류 평균 -> 시드 평균, 단위 %
from sklearn.metrics import roc_auc_score

rows = []
splits = {c: data.list_mvtec2(LOCAL, c) for c in CATEGORIES}
for r in pd.read_csv(CSV).itertuples():
    stem = f"mvtec_ad_2__{r.category}__{r.setting}__compare__nall__s{r.seed}"
    saved = np.load(f"{RESULT_DIR}/q1_scores/{stem}.npz")
    test = splits[r.category].test
    labels = np.array([s.label for s in test])
    assert np.array_equal(labels, saved["labels"]), stem  # 저장 순서와 목록 순서가 같다
    cond = np.array([s.anomaly for s in test])
    for c in sorted(set(cond)):
        pick = cond == c
        rows.append({"setting": r.setting, "category": r.category, "seed": r.seed, "condition": c,
                     "image_auroc": roc_auc_score(labels[pick], saved["scores"][pick])})
by_cond = pd.DataFrame(rows)
table = (by_cond.groupby(["setting", "condition", "seed"]).image_auroc.mean()
         .groupby(["setting", "condition"]).mean().unstack() * 100)
display(table.round(2))
print("조건마다 든 부류 수:", by_cond.groupby("condition").category.nunique().to_dict())
print("regular 만으로 낸 순위와 전체 순위의 Kendall tau-b")
display(ex.kendall_pairs(pd.DataFrame({"all": values["mvtec_ad_2"], "regular": table["regular"],
                                       "mvtec_ad": values["mvtec_ad"]})).round(3))


condition,overexposed,regular,shift_1,shift_2,shift_3,shift_4,underexposed
setting,,,,,,,
padim_r18_L123_d100,64.15,65.65,66.71,63.83,65.17,78.67,64.30
padim_r18_L23_d100,64.04,65.86,63.10,58.29,61.59,88.89,62.48
padim_wrn50_L123_d550,62.88,65.17,67.88,60.84,65.65,81.33,65.09
padim_wrn50_L23_d550,64.81,69.57,67.03,63.96,66.14,83.56,68.36
patchcore_r18_L123_c1,65.43,67.83,69.74,61.74,65.96,83.56,69.97
patchcore_r18_L23_c1,65.15,66.75,68.58,62.43,65.15,80.44,67.40
patchcore_r18_L23_c10,61.29,62.66,65.23,60.48,62.58,88.44,63.55
patchcore_wrn50_L123_c1,68.79,70.03,69.34,62.60,67.75,83.11,70.24
patchcore_wrn50_L23_c1,71.66,72.82,75.90,67.32,68.39,95.11,69.12


조건마다 든 부류 수: {np.str_('overexposed'): 8, np.str_('regular'): 8, np.str_('shift_1'): 8, np.str_('shift_2'): 7, np.str_('shift_3'): 7, np.str_('shift_4'): 1, np.str_('underexposed'): 8}
regular 만으로 낸 순위와 전체 순위의 Kendall tau-b


,pair,n_settings,n_pairs,concordant,discordant,tied,tau_b,p_value
0,all - regular,10,45,39,6,0,0.733,0.002
1,all - mvtec_ad,10,45,36,9,0,0.600,0.017
2,regular - mvtec_ad,10,45,34,11,0,0.511,0.047


## 6. 결과를 레포로

Drive `beyond_mvtec_results/`의 `q1_mvtec_ad_2.csv`를 레포 `results/`에 넣는다. `q1_scores/`와 `mvtec2_256/`은 올리지 않는다.